In [1]:
import subprocess
result = subprocess.run(
    ["pip", "install", "dbt-snowflake", "-q"],
    capture_output=True, text=True
)
print("✅ dbt-snowflake installed")

# Verify
result = subprocess.run(["dbt", "--version"], capture_output=True, text=True)
print(result.stdout)

✅ dbt-snowflake installed
Core:
  - installed: 1.11.11
  - latest:    1.11.11 - Up to date!

Plugins:
  - snowflake: 1.11.5 - Up to date!





cd /home/jovyan/work
dbt init ipl_dbt --skip-profile-creation
cd ipl_dbt

Step 3 — Create profiles.yml
Create file at /home/jovyan/work/ipl_dbt/profiles.yml:
yamlipl_dbt:
  target: dev
  outputs:
    dev:
      type: snowflake
      account: "{{ env_var('SNOWFLAKE_ACCOUNT') }}"
      user: "{{ env_var('SNOWFLAKE_USER') }}"
      password: "{{ env_var('SNOWFLAKE_PASSWORD') }}"
      database: IPL_DB
      schema: STAGING
      warehouse: IPL_WH
      threads: 1

Step 4 — Create dbt models
Create /home/jovyan/work/ipl_dbt/models/staging/stg_team_wins.sql:
sql-- Staging model: clean and rename raw data
SELECT
    winner                              AS team_name,
    total_wins,
    ROUND(total_wins * 100.0 / SUM(total_wins) OVER (), 2) AS win_percentage
FROM {{ source('ipl_raw', 'team_wins') }}
WHERE winner IS NOT NULL
Create /home/jovyan/work/ipl_dbt/models/staging/stg_season_summary.sql:
sqlSELECT
    season,
    matches_played,
    LAG(matches_played) OVER (ORDER BY season) AS prev_season_matches,
    matches_played - LAG(matches_played) OVER (ORDER BY season) AS matches_growth
FROM {{ source('ipl_raw', 'season_summary') }}
Create /home/jovyan/work/ipl_dbt/models/mart/mart_team_performance.sql:
sql-- Final mart model: team performance summary
SELECT
    team_name,
    total_wins,
    win_percentage,
    RANK() OVER (ORDER BY total_wins DESC) AS performance_rank,
    CASE
        WHEN win_percentage >= 15 THEN 'Elite'
        WHEN win_percentage >= 10 THEN 'Strong'
        WHEN win_percentage >= 5  THEN 'Average'
        ELSE 'Developing'
    END AS performance_tier
FROM {{ ref('stg_team_wins') }}
ORDER BY total_wins DESC
Create /home/jovyan/work/ipl_dbt/models/sources.yml:
yamlversion: 2

sources:
  - name: ipl_raw
    database: IPL_DB
    schema: RAW
    tables:
      - name: team_wins
      - name: season_summary
      - name: toss_analysis

models:
  - name: stg_team_wins
    description: "Cleaned team wins with win percentage"
    columns:
      - name: team_name
        tests: [not_null, unique]
      - name: total_wins
        tests: [not_null]

  - name: mart_team_performance
    description: "Final mart with ranking and tiers"
    columns:
      - name: team_name
        tests: [not_null, unique]
      - name: performance_rank
        tests: [not_null]

Step 5 — Run dbt
bashcd /home/jovyan/work/ipl_dbt

# Test connection
dbt debug

# Run all models
dbt run

# Run tests
dbt test

# See lineage docs
dbt docs generate
dbt docs serve --port 8080
You should see:
✅ stg_team_wins  .....  SUCCESS
✅ stg_season_summary  . SUCCESS
✅ mart_team_performance SUCCESS